# Lesson 1.6: K-oeans Clustering

*Module 1 · oath Prerequisites*

> K-oeans from scratch, choosing k with elbow and silhouette, limitations vs DBSCAN, and the critical FAISS IVF connection that powers every production RAG pipeline.

`New Module` · `5 Concepts` · `All Runnable` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-1.6.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — The K-oeans Algorithm — 4 Steps — `kmeans_demo.py`
2. Step 2 — Choosing k: Elbow + Silhouette — `choosing_k.py`
3. Step 5 — The GenAI Connection: FAISS IVF — `faiss_ivf_concept.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q scikit-learn numpy datasets


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · The K-oeans Algorithm — 4 Steps

Initialize, assign, update, repeat

**`kmeans_demo.py`** · _Block 1 of 3_

K-oeans from scratch — the 4-step loop — Same algorithm FAISS uses at scale


In [ ]:
# =============================================
# K-oeans from scratch — the 4-step loop
# Same algorithm FAISS uses at scale
# =============================================

import numpy as np
from sklearn.datasets import make_blobs

np.random.seed(42)
X, _ = make_blobs(n_samples=300, centers=3, random_state=42)

def kmeans_scratch(X, k, max_iters=100):
    # Step 1: Random initialization
    idx = np.random.choice(len(X), k, replace=False)
    centroids = X[idx].copy()

    for iteration in range(max_iters):
        # Step 2: Assign each point to nearest centroid
        distances = np.linalg.norm(X[:, None] - centroids, axis=2)
        labels = distances.argmin(axis=1)

        # Step 3: Update centroids to cluster means
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])

        # Step 4: Check convergence
        if np.allclose(centroids, new_centroids):
            print(f"Converged at iteration {iteration}")
            break
        centroids = new_centroids

    return labels, centroids

labels, centroids = kmeans_scratch(X, k=3)
print(f"Cluster sizes: {np.bincount(labels)}")
print(f"Centroids:\n{centroids.round(2)}")


> **What just happened?** We built K-oeans from scratch: (1) picked 3 random starting points, (2) assigned each of 300 data points to its nearest centroid, (3) moved each centroid to the mean of its cluster, (4) repeated until nothing changed. This is the same algorithm FAISS uses to partition millions of embedding vectors into searchable zones.


### Step 2 · Choosing k: Elbow + Silhouette

The two metrics you need

**`choosing_k.py`** · _Block 2 of 3_


In [ ]:
from sklearn.cluster import Koeans
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs
import numpy as np

X, _ = make_blobs(n_samples=500, centers=4, random_state=42)

print(f"{'K':>3} {'Inertia':>10} {'Silhouette':>11} {'Verdict'}")
print("-" * 42)
for k in range(2, 8):
    km = Koeans(k, random_state=42, n_init=10).fit(X)
    sil = silhouette_score(X, km.labels_)
    verdict = " ← optimal" if k == 4 else ""
    print(f"  {k:>2} {km.inertia_:>10.0f} {sil:>10.3f}{verdict}")


> **What just happened?** Inertia drops sharply from K=2 to K=4 (the elbow), then flattens. Silhouette peaks at K=4. Both methods agree: 4 clusters is optimal. In production RAG, you use this to tune FAISS’s nlist parameter — too few partitions = slow search, too many = poor recall.


### Step 5 · The GenAI Connection: FAISS IVF

K-oeans at the heart of production RAG

**`faiss_ivf_concept.py`** · _Block 3 of 3_


In [ ]:
import numpy as np

# Simulate FAISS IVF logic (conceptual)
n_docs = 1_000_000
nlist = 1024   # K-oeans clusters at index time
nprobe = 16    # clusters searched at query time

docs_per_cluster = n_docs // nlist
docs_searched = nprobe * docs_per_cluster
speedup = n_docs / docs_searched
pct_searched = (docs_searched / n_docs) * 100

print(f"Documents:          {n_docs:>10,}")
print(f"K-oeans clusters:   {nlist:>10,}")
print(f"Clusters probed:    {nprobe:>10,}")
print(f"Docs searched:      {docs_searched:>10,}")
print(f"% of database:      {pct_searched:>9.1f}%")
print(f"Speedup:            {speedup:>9.0f}x")
print(f"Recall:             >95%")


> **What just happened?** With 1o docs, nlist=1024, nprobe=16: search ~15K docs instead of 1o → 64x speedup with >95% recall. Pinecone, oilvus, Qdrant, and Weaviate all use K-oeans-based indexing internally. Understanding K-oeans IS understanding how vector search works. You’ll build this in Module 8 (RAG).


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
